# Deploy Deny Policy for Model Deployments

Apply an Azure Policy to **prevent model deployments** in application team (spoke) resource groups.

## Why This Policy?

| Without Policy | With Policy |
|----------------|-------------|
| Teams can deploy their own models | All models centralised in the core gateway |
| Fragmented cost tracking | Unified cost management |
| Inconsistent security | Consistent content filters & rate limits |
| No visibility | Full observability via APIM |


> **Hub gateway** = `rg-foundry-core-{suffix}` — the shared APIM instance and hub Foundry accounts with all model deployments. Spoke resource groups connect to it but hold no models of their own.

## Prerequisites
The hub, team alpha spoke, and multi project spokes are provisioned.

## Step 1: Define the Policy

In [1]:
import json

# Policy rule: Deny CognitiveServices model deployments
# Note: The "mode" is specified via --mode flag, not in the rules JSON
POLICY_RULE = {
    "if": {
        "field": "type",
        "equals": "Microsoft.CognitiveServices/accounts/deployments"
    },
    "then": {
        "effect": "deny"
    }
}

print("Policy Rule:")
print(json.dumps(POLICY_RULE, indent=2))

Policy Rule:
{
  "if": {
    "field": "type",
    "equals": "Microsoft.CognitiveServices/accounts/deployments"
  },
  "then": {
    "effect": "deny"
  }
}


## Step 2: Derive Variables

In [2]:
import subprocess, hashlib, os
from pathlib import Path
from dotenv import load_dotenv

# Derive stable 6-char suffix from subscription ID (same as Labs 1A-1D)
SUBSCRIPTION_ID = subprocess.run(
    'az account show --query id -o tsv',
    shell=True, capture_output=True, text=True
).stdout.strip()
SUFFIX = hashlib.sha256((SUBSCRIPTION_ID + 'v2').encode()).hexdigest()[:6]

# Load .env from repo root
repo_root = Path(subprocess.run(
    'git rev-parse --show-toplevel', shell=True, capture_output=True, text=True
).stdout.strip())
load_dotenv(repo_root / '.env', override=True)

# Resource groups — names match Labs 1A-1D naming conventions
SPOKE_RG = f"rg-foundry-spoke-alpha-{SUFFIX}"   # Team Alpha 1:1 spoke (Lab 1B)
MULTI_RG = f"rg-foundry-multi-{SUFFIX}"          # Teams Beta/Delta/Gamma 1:N (Lab 1C)
LZ_RG    = f"rg-foundry-core-{SUFFIX}"            # Landing Zone — must NOT be blocked

POLICY_NAME = "deny-model-deployments"

print(f"Subscription: {SUBSCRIPTION_ID}")
print(f"Suffix:       {SUFFIX}")
print(f"Spoke RG:     {SPOKE_RG}")
print(f"Multi RG:     {MULTI_RG}")
print(f"LZ RG:        {LZ_RG}  (excluded — models live here)")
print(f"Policy Name:  {POLICY_NAME}")

Subscription: 025aba94-0c4a-443d-8826-466477e2850f
Suffix:       c2676f
Spoke RG:     rg-foundry-spoke-alpha-c2676f
Multi RG:     rg-foundry-multi-c2676f
LZ RG:        rg-foundry-core-c2676f  (excluded — models live here)
Policy Name:  deny-model-deployments


## Step 3: Create Policy Definition at Subscription Level

In [3]:
# Write policy rule to temp file (only the if/then block, not mode)
import tempfile
import os

policy_file = tempfile.NamedTemporaryFile(mode='w', suffix='.json', delete=False)
json.dump(POLICY_RULE, policy_file)
policy_file.close()

# Create policy definition
!az policy definition create \
    --name "{POLICY_NAME}" \
    --display-name "Deny AI Model Deployments" \
    --description "Prevents deployment of AI models in spoke resource groups. All models must be deployed in the central Landing Zone." \
    --rules "{policy_file.name}" \
    --mode All \
    -o table

os.unlink(policy_file.name)
print("\n✅ Policy definition created")

Description                                                                                                          DisplayName                Mode    Name                    PolicyType    Version
-------------------------------------------------------------------------------------------------------------------  -------------------------  ------  ----------------------  ------------  ---------
Prevents deployment of AI models in spoke resource groups. All models must be deployed in the central Landing Zone.  Deny AI Model Deployments  All     deny-model-deployments  Custom        1.0.0

✅ Policy definition created


## Step 4: Assign Policy to Spoke Resource Groups

The policy is assigned at **resource group level** to each spoke, leaving the core gateway unaffected.

| Resource group | Provisioned by | Policy |
|---|---|---|
| `rg-foundry-core-{suffix}` | `05-02-deploy-foundry-core-gateway` | ✅ Allowed — models deployed here |
| `rg-foundry-spoke-alpha-{suffix}` | `05-03-deploy-foundry-project-spoke` | ❌ Blocked |
| `rg-foundry-multi-{suffix}` | `05-04-deploy-foundry-multi-project` | ❌ Blocked (skipped if not deployed) |

In [4]:
POLICY_DEF_ID = f"/subscriptions/{SUBSCRIPTION_ID}/providers/Microsoft.Authorization/policyDefinitions/{POLICY_NAME}"

# Determine which RGs exist — MULTI_RG is only present if Lab 1C has been run
rg_check = subprocess.run(
    f'az group exists -n "{MULTI_RG}"',
    shell=True, capture_output=True, text=True
).stdout.strip()
target_rgs = [SPOKE_RG] + ([MULTI_RG] if rg_check == "true" else [])

if rg_check != "true":
    print(f"ℹ️  {MULTI_RG} not found — skipping (run Lab 1C to include it)\n")

for rg in target_rgs:
    scope           = f"/subscriptions/{SUBSCRIPTION_ID}/resourceGroups/{rg}"
    assignment_name = f"{POLICY_NAME}-{rg}"

    result = subprocess.run(
        f'az policy assignment create '
        f'--name "{assignment_name}" '
        f'--display-name "Deny Model Deployments in {rg}" '
        f'--policy "{POLICY_DEF_ID}" '
        f'--scope "{scope}" '
        f'-o table',
        shell=True, capture_output=True, text=True
    )
    print(result.stdout.strip())
    if result.returncode == 0:
        print(f"✅ Policy assigned to {rg}\n")
    else:
        print(f"❌ Failed for {rg}: {result.stderr[:300]}\n")

DefinitionVersion    DisplayName                                              EnforcementMode    Name                                                  PolicyDefinitionId                                                                                                              ResourceGroup                  Scope
-------------------  -------------------------------------------------------  -----------------  ----------------------------------------------------  ------------------------------------------------------------------------------------------------------------------------------  -----------------------------  ------------------------------------------------------------------------------------------------
1.*.*                Deny Model Deployments in rg-foundry-spoke-alpha-c2676f  Default            deny-model-deployments-rg-foundry-spoke-alpha-c2676f  /subscriptions/025aba94-0c4a-443d-8826-466477e2850f/providers/Microsoft.Authorization/policyDefinitions/deny-model-deployments

## Step 5: Test the Policy

Try to deploy a model in the spoke - it should fail!

In [5]:
# Env vars written by Labs 1A and 1B
ALPHA_ACCOUNT = os.environ.get('ALPHA_FOUNDRY_ACCOUNT', '')
CHAT_MODEL    = os.environ.get('CHAT_MODEL', 'gpt-4.1-mini')
MODEL_VERSION = '2025-04-14'  # gpt-4.1-mini GA version

if ALPHA_ACCOUNT:
    print(f"Attempting to deploy a model to spoke account: {ALPHA_ACCOUNT}")
    print("This should FAIL due to the policy...\n")

    result = subprocess.run(
        f'az cognitiveservices account deployment create '
        f'-g "{SPOKE_RG}" '
        f'-n "{ALPHA_ACCOUNT}" '
        f'--deployment-name "test-blocked" '
        f'--model-name "{CHAT_MODEL}" '
        f'--model-version "{MODEL_VERSION}" '
        f'--model-format OpenAI '
        f'--sku-name GlobalStandard '
        f'--sku-capacity 1',
        shell=True, capture_output=True, text=True
    )

    if "RequestDisallowedByPolicy" in result.stderr or "denied by policy" in result.stderr.lower():
        print("✅ SUCCESS! Deployment was blocked by policy:")
        print("   'RequestDisallowedByPolicy' — model deployments are denied in spoke.")
    elif result.returncode != 0:
        print(f"❌ Error (check if policy related): {result.stderr[:500]}")
    else:
        print("⚠️  Deployment succeeded — policy may not be active yet (can take a few minutes)")
else:
    print("⚠️  ALPHA_FOUNDRY_ACCOUNT not found in .env — run Lab 1B first")

Attempting to deploy a model to spoke account: aif-spoke-alpha-c2676f
This should FAIL due to the policy...

✅ SUCCESS! Deployment was blocked by policy:
   'RequestDisallowedByPolicy' — model deployments are denied in spoke.


## Step 6: Verify Spoke Can Still USE Models

The policy blocks **deployments**, but using models via APIM connection should still work.

In [6]:
import re
from azure.identity import DefaultAzureCredential
from azure.ai.projects import AIProjectClient
from azure.ai.projects.models import PromptAgentDefinition

def extract_response(text):
    """Strip DeepSeek <think> tags and leading whitespace."""
    text = re.sub(r'<think>.*?</think>', '', text, flags=re.DOTALL)
    return text.strip()

# Env vars written by Labs 1A and 1B
PROJECT_ENDPOINT = os.environ.get('ALPHA_FOUNDRY_PROJECT_ENDPOINT', '')
CORE_CONNECTION   = os.environ.get('ALPHA_FOUNDRY_CORE_CONNECTION', '')
CHAT_MODEL       = os.environ.get('CHAT_MODEL', 'gpt-4.1-mini')

if PROJECT_ENDPOINT and CORE_CONNECTION:
    gateway_model = f"{CORE_CONNECTION}/{CHAT_MODEL}"   # e.g. "core-alpha/gpt-4.1-mini"
    agent_name    = "policy-test-agent"

    print(f"Testing: use LZ model via Foundry Agent API")
    print(f"Project endpoint: {PROJECT_ENDPOINT}")
    print(f"Gateway model:    {gateway_model}")
    print()

    credential = DefaultAzureCredential()
    client     = AIProjectClient(credential=credential, endpoint=PROJECT_ENDPOINT)
    openai     = client.get_openai_client()

    try:
        agent = client.agents.create_version(
            agent_name=agent_name,
            definition=PromptAgentDefinition(model=gateway_model, instructions="Be brief.")
        )
        resp = openai.responses.create(
            input="Say 'Policy test passed!' in 5 words or less",
            extra_body={"agent_reference": {"name": agent.name, "version": agent.version, "type": "agent_reference"}}
        )
        text = extract_response(resp.output_text)
        print(f"✅ SUCCESS! Agent responded: {text}")
        print()
        print("Summary:")
        print("   🚫 Model DEPLOYMENT in spoke: BLOCKED by policy")
        print("   ✅ Model USAGE via Agent API: ALLOWED (different resource type)")
    except Exception as e:
        print(f"❌ Error: {e}")
    finally:
        client.agents.delete(agent_name=agent_name)
        openai.close()
else:
    print("⚠️  ALPHA_FOUNDRY_PROJECT_ENDPOINT or ALPHA_FOUNDRY_CORE_CONNECTION not found in .env")
    print("    Run Labs 1A and 1B first")

Testing: use LZ model via Foundry Agent API
Project endpoint: https://aif-spoke-alpha-c2676f.services.ai.azure.com/api/projects/project-alpha-c2676f
Gateway model:    core-alpha/gpt-4.1-mini

✅ SUCCESS! Agent responded: Policy test passed!

Summary:
   🚫 Model DEPLOYMENT in spoke: BLOCKED by policy
   ✅ Model USAGE via Agent API: ALLOWED (different resource type)


## Done!

The governance policy is now in place:

| Resource Group | Provisioned by | Model Deployments |
|---|---|---|
| `rg-foundry-core-{suffix}` | `05-02-deploy-foundry-core-gateway` | ✅ Allowed |
| `rg-foundry-spoke-alpha-{suffix}` | `05-03-deploy-foundry-project-spoke` | ❌ Blocked by Policy |
| `rg-foundry-multi-{suffix}` | `05-04-deploy-foundry-multi-project` | ❌ Blocked by Policy |

### Extending to Additional Spokes

To apply the policy to any future spoke resource group:

```bash
az policy assignment create \
    --name "deny-model-deployments-<spoke-rg>" \
    --policy "<policy-definition-id>" \
    --scope "/subscriptions/<sub-id>/resourceGroups/<spoke-rg>"
```

Or assign at **subscription level** with an exclusion for `rg-foundry-core-{suffix}` — this automatically covers all current and future spoke resource groups without per-RG assignments.

**Next**: Build agents with the Agent Service

## Cleanup (Optional)

In [7]:
# Remove policy assignments and definition
# Run each line individually after confirming the resource group name

# Team Alpha spoke
# !az policy assignment delete --name "{POLICY_NAME}-{SPOKE_RG}" --scope "/subscriptions/{SUBSCRIPTION_ID}/resourceGroups/{SPOKE_RG}"

# Multi-project spoke (if Lab 1C was run)
# !az policy assignment delete --name "{POLICY_NAME}-{MULTI_RG}" --scope "/subscriptions/{SUBSCRIPTION_ID}/resourceGroups/{MULTI_RG}"

# Policy definition (subscription-level)
# !az policy definition delete --name "{POLICY_NAME}"

# print("✅ Policy removed")